# Week 4: Major Project (Capstone Project)
## Customer Segmentation using Unsupervised K-Means Clustering
**Dataset**: Mall Customers Dataset (Kaggle)  
**Focus**: Full Machine Learning Pipeline + Statistical Profiling + Interactive Visualizations + Business Strategy & Presentation

---
### 📌 Project Executive Summary
In retail and commercial mall management, understanding customer behavioral dynamics is essential for targeted marketing, inventory optimization, tenant leasing, and maximizing Customer Lifetime Value (CLV).

Rather than treating all shoppers homogeneously, **Customer Segmentation** employs unsupervised machine learning to group consumers based on shared demographic and spending characteristics:
- **Age**: Captures generational spending power, lifestyle, and brand preferences.
- **Annual Income ($k)**: Reflects consumer purchasing capacity and disposable budget.
- **Spending Score (1–100)**: Proprietary mall metric quantifying shopper purchase frequency, basket size, and behavioral receptiveness to promotions.

---
### 🎯 Capstone Objectives & Workflow Requirements (Option 3):
1. **Apply K-Means Clustering** to segment mall patrons into mathematically distinct clusters.
2. **Determine Optimal Clusters ($K$)** using the **Elbow Method (WCSS)** and **Silhouette Coefficient Analysis**.
3. **Multi-Dimensional Visualization**: Visualize customer groups across **Age, Annual Income, and Spending Score** in both 2D and 3D coordinate spaces.
4. **Group Labeling**: Categorize clusters into core spending tiers:
   - 🟢 **High Spenders**
   - 🟡 **Medium Spenders**
   - 🔴 **Low Spenders**
   alongside granular actionable consumer personas (*Affluent VIPs, Young Enthusiasts, Careful Savers, Budget Conscious, Mainstream Middle*).
5. **Business Analytics & Retail Strategies**: Deliver ROI-driven promotional, loyalty, and tenant allocation playbooks for mall leadership.
6. **Production Model Serialization**: Save and validate trained estimators and preprocessors using `joblib`.


---
## Step 1: Environment Setup & Data Ingestion
Load necessary scientific computing, clustering, and visualization libraries.


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples, calinski_harabasz_score, davies_bouldin_score

# Set plot styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

# Load Dataset
data_path = 'Mall_Customers.csv' if os.path.exists('Mall_Customers.csv') else 'week4/Mall_Customers.csv'
df = pd.read_csv(data_path)

# Standardize column naming
df.rename(columns={
    'CustomerID': 'CustomerID',
    'Genre': 'Gender',
    'Age': 'Age',
    'Annual Income (k$)': 'Annual_Income_k$',
    'Spending Score (1-100)': 'Spending_Score_1_100'
}, inplace=True)

print(f"Dataset successfully loaded with {df.shape[0]} rows and {df.shape[1]} features.")
df.head(10)


---
## Step 2: Exploratory Data Analysis (EDA) & Demographic Profiling
Verify missing values, inspect distribution percentiles, and analyze gender-based spending disparities.


In [ ]:
# Dataset schema and null value inspection
print("Data Types & Missing Value Audit:")
print(df.info())
print("
Missing Values Count:")
print(df.isnull().sum())

# Statistical summary
print("
Numerical Feature Summary Statistics:")
df[['Age', 'Annual_Income_k$', 'Spending_Score_1_100']].describe().T


In [ ]:
# Gender Breakdown & Spending Comparison
gender_summary = df.groupby('Gender').agg(
    Customer_Count=('CustomerID', 'count'),
    Mean_Age=('Age', 'mean'),
    Mean_Annual_Income=('Annual_Income_k$', 'mean'),
    Mean_Spending_Score=('Spending_Score_1_100', 'mean')
).reset_index()

gender_summary['Percentage'] = (gender_summary['Customer_Count'] / len(df)) * 100
gender_summary.round(2)


In [ ]:
# Visualizing Feature Distributions & Correlations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age Distribution
sns.histplot(data=df, x='Age', hue='Gender', kde=True, bins=15, ax=axes[0], palette={'Male': '#3498db', 'Female': '#e74c3c'})
axes[0].set_title('Age Distribution by Gender', fontsize=12, fontweight='bold')

# Income Distribution
sns.histplot(data=df, x='Annual_Income_k$', hue='Gender', kde=True, bins=15, ax=axes[1], palette={'Male': '#3498db', 'Female': '#e74c3c'})
axes[1].set_title('Annual Income Distribution by Gender', fontsize=12, fontweight='bold')

# Spending Score Distribution
sns.histplot(data=df, x='Spending_Score_1_100', hue='Gender', kde=True, bins=15, ax=axes[2], palette={'Male': '#3498db', 'Female': '#e74c3c'})
axes[2].set_title('Spending Score Distribution by Gender', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Correlation Matrix
plt.figure(figsize=(7, 5))
corr = df[['Age', 'Annual_Income_k$', 'Spending_Score_1_100']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.3f', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.show()


---
## Step 3: Determining Optimal Number of Clusters ($K$)
### Mathematical Formulation:
1. **Within-Cluster Sum of Squares (WCSS / Inertia)**:
   $$WCSS = \sum_{i=1}^{k} \sum_{x \in S_i} ||x - \mu_i||^2$$
   Measures the compactness of clusters. The "Elbow" represents the inflection point beyond which adding more clusters yields diminishing returns in variance reduction.

2. **Silhouette Coefficient**:
   $$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$
   Where $a(i)$ is mean intra-cluster distance and $b(i)$ is mean nearest-cluster distance. Ranges from $-1$ to $+1$, where values $>0.5$ indicate well-separated and dense clusters.


In [ ]:
# Extract 2D Feature Space: Annual Income & Spending Score
X_2d = df[['Annual_Income_k$', 'Spending_Score_1_100']].values

k_range = list(range(2, 11))
inertia_list = []
silhouette_list = []
calinski_list = []
davies_list = []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=25, max_iter=300, random_state=42)
    labels = km.fit_predict(X_2d)
    
    inertia_list.append(km.inertia_)
    silhouette_list.append(silhouette_score(X_2d, labels))
    calinski_list.append(calinski_harabasz_score(X_2d, labels))
    davies_list.append(davies_bouldin_score(X_2d, labels))

# Plot Elbow and Silhouette curves side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Elbow Curve
ax1.plot(k_range, inertia_list, marker='o', color='#2c3e50', linewidth=2.5)
ax1.axvline(5, color='#e74c3c', linestyle='--', label='Elbow Point (K=5)')
ax1.set_title('Elbow Method (WCSS vs K)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia (WCSS)')
ax1.legend()

# Silhouette Score Curve
ax2.plot(k_range, silhouette_list, marker='s', color='#16a085', linewidth=2.5)
ax2.plot(5, silhouette_list[k_range.index(5)], marker='*', color='#e74c3c', markersize=15, label=f'Peak Score = {silhouette_list[k_range.index(5)]:.3f} (K=5)')
ax2.set_title('Silhouette Analysis vs K', fontsize=13, fontweight='bold')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.legend()

plt.tight_layout()
plt.show()

# Tabular display of metrics
metrics_df = pd.DataFrame({
    'K': k_range,
    'Inertia': inertia_list,
    'Silhouette Score': silhouette_list,
    'Calinski-Harabasz': calinski_list,
    'Davies-Bouldin': davies_list
})
print("Clustering Validation Metrics Summary:")
metrics_df.round(4)


---
## Step 4: K-Means Model Training & Persona Profiling ($K=5$)
Fit the final optimal K-Means model on Annual Income and Spending Score, extract cluster centroids, and establish business profiles.


In [ ]:
# Fit Final K-Means Model
optimal_k = 5
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', n_init=25, max_iter=300, random_state=42)
df['Cluster_ID'] = kmeans.fit_predict(X_2d)

# Calculate Cluster Centroids & Profiles
centroids = kmeans.cluster_centers_

# Assign Spending Groups ('Low Spenders', 'Medium Spenders', 'High Spenders') & Personas
def assign_segment(row, centroids):
    c_id = row['Cluster_ID']
    income_center = centroids[c_id, 0]
    spend_center = centroids[c_id, 1]
    
    # PDF Tier Requirements
    if spend_center >= 65:
        tier = "High Spenders"
        persona = "Affluent VIPs" if income_center > 60 else "Young Enthusiasts"
    elif spend_center <= 35:
        tier = "Low Spenders"
        persona = "Careful Savers" if income_center > 60 else "Budget Conscious"
    else:
        tier = "Medium Spenders"
        persona = "Mainstream Middle"
        
    return pd.Series([tier, persona])

df[['Spending_Group', 'Persona']] = df.apply(lambda r: assign_segment(r, centroids), axis=1)

# Group profiling summary
cluster_summary = df.groupby('Cluster_ID').agg(
    Persona=('Persona', 'first'),
    Spending_Group=('Spending_Group', 'first'),
    Customer_Count=('CustomerID', 'count'),
    Mean_Income=('Annual_Income_k$', 'mean'),
    Mean_Spend=('Spending_Score_1_100', 'mean'),
    Mean_Age=('Age', 'mean')
).reset_index()

cluster_summary['Market_Share_%'] = (cluster_summary['Customer_Count'] / len(df)) * 100
cluster_summary.round(2)


---
## Step 5: Visualizing Customer Groups Across Age, Income, & Spending Score
### 1. 2D Cluster Map (Annual Income vs Spending Score)
Scatter plot showing individual customers, cluster colors, centroids, and spending tiers.


In [ ]:
# 2D Segmentation Scatter Plot
plt.figure(figsize=(12, 8))
palette = ['#2ecc71', '#3498db', '#e74c3c', '#9b59b6', '#f39c12']

for c_id in sorted(df['Cluster_ID'].unique()):
    c_data = df[df['Cluster_ID'] == c_id]
    persona = c_data['Persona'].iloc[0]
    group = c_data['Spending_Group'].iloc[0]
    plt.scatter(c_data['Annual_Income_k$'], c_data['Spending_Score_1_100'],
                s=80, alpha=0.85, label=f"Cluster {c_id}: {persona} [{group}]",
                edgecolor='white', linewidth=0.8)

# Overlay Centroids
plt.scatter(centroids[:, 0], centroids[:, 1], s=300, c='black', marker='X', edgecolors='gold', linewidths=2, label='Cluster Centroids', zorder=10)

# Annotate Centroids
for i, c in enumerate(centroids):
    p_name = df[df['Cluster_ID'] == i]['Persona'].iloc[0]
    plt.annotate(f"C{i}: {p_name}\n(${c[0]:.0f}k, {c[1]:.0f})", (c[0] + 1.5, c[1] - 3.5),
                 fontsize=9.5, fontweight='bold', bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.6, ec="black"))

plt.title('Mall Customer Segmentation Map (Income vs Spending Score)', fontsize=15, fontweight='bold')
plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Spending Score (1-100)', fontsize=12)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
plt.tight_layout()
plt.show()


In [ ]:
# Visualizing Spending Score and Age across Customer Personas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Spending Score by Persona
sns.boxplot(data=df, x='Persona', y='Spending_Score_1_100', hue='Spending_Group', ax=ax1, palette='Set2')
ax1.set_title('Spending Score Distribution by Persona', fontsize=12, fontweight='bold')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=25, ha='right')

# Age Distribution by Persona
sns.boxplot(data=df, x='Persona', y='Age', hue='Spending_Group', ax=ax2, palette='Set2')
ax2.set_title('Age Distribution Across Personas', fontsize=12, fontweight='bold')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=25, ha='right')

plt.tight_layout()
plt.show()


### 2. 3D Customer Segmentation: Age vs Annual Income vs Spending Score
Examine the interaction of customer age with purchasing power and spending propensity.


In [ ]:
# 3D Projection
fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection='3d')

colors = ['#2ecc71', '#3498db', '#e74c3c', '#9b59b6', '#f39c12']

for c_id in sorted(df['Cluster_ID'].unique()):
    sub = df[df['Cluster_ID'] == c_id]
    persona = sub['Persona'].iloc[0]
    group = sub['Spending_Group'].iloc[0]
    ax.scatter(sub['Age'], sub['Annual_Income_k$'], sub['Spending_Score_1_100'],
               c=colors[c_id], label=f"C{c_id}: {persona} [{group}]",
               s=60, alpha=0.85, edgecolors='w', depthshade=True)

ax.set_title('3D Customer Segmentation: Age vs Income vs Spending Score', fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Age (Years)', fontsize=10, labelpad=8)
ax.set_ylabel('Annual Income (k$)', fontsize=10, labelpad=8)
ax.set_zlabel('Spending Score (1-100)', fontsize=10, labelpad=8)
ax.view_init(elev=25, azim=130)
ax.legend(bbox_to_anchor=(1.15, 0.9), loc='upper left')

plt.tight_layout()
plt.show()


---
## Step 6: Macro Spending Tiers Analysis ("Low Spenders", "Medium Spenders", "High Spenders")
Compare the 5 micro-clusters with the direct 3-tier macro categorization.


In [ ]:
# Spending Tier Breakdown
group_breakdown = df.groupby('Spending_Group').agg(
    Customer_Count=('CustomerID', 'count'),
    Avg_Age=('Age', 'mean'),
    Avg_Income=('Annual_Income_k$', 'mean'),
    Avg_Spending_Score=('Spending_Score_1_100', 'mean')
).loc[['High Spenders', 'Medium Spenders', 'Low Spenders']]

group_breakdown['Market_Share_%'] = (group_breakdown['Customer_Count'] / len(df)) * 100
print("Overarching Spending Groups Summary:")
group_breakdown.round(2)


In [ ]:
# Spending Group Market Share Pie and Bar Charts
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Donut Chart
colors_tier = {'High Spenders': '#e74c3c', 'Medium Spenders': '#f1c40f', 'Low Spenders': '#3498db'}
group_counts = df['Spending_Group'].value_counts()[['High Spenders', 'Medium Spenders', 'Low Spenders']]

ax1.pie(group_counts, labels=group_counts.index, autopct='%1.1f%%', startangle=140,
        colors=[colors_tier[k] for k in group_counts.index], wedgeprops=dict(width=0.4, edgecolor='white'))
ax1.set_title('Customer Base Proportion by Spending Group', fontsize=13, fontweight='bold')

# Bar Chart
sns.barplot(x=group_counts.index, y=group_counts.values, palette=colors_tier, ax=ax2, edgecolor='black')
for i, v in enumerate(group_counts.values):
    ax2.text(i, v + 2, f"{v} ({v/len(df)*100:.1f}%)", ha='center', fontweight='bold')
ax2.set_title('Customer Count by Spending Tier', fontsize=13, fontweight='bold')
ax2.set_ylabel('Number of Customers')
ax2.set_ylim(0, max(group_counts.values) * 1.15)

plt.tight_layout()
plt.show()


---
## Step 7: Model Persistence & Inference Engine
Save trained estimator and build a production inference function for newly registered shoppers.


In [ ]:
# Save trained model
model_path = 'kmeans_customer_model.joblib'
scaler_path = 'scaler.joblib'

joblib.dump(kmeans, model_path)
print(f"Model saved successfully to '{model_path}'")

# Real-time Scoring Function
def predict_customer_segment(income_k, spend_score, model):
    pred_cluster = model.predict([[income_k, spend_score]])[0]
    
    tier_map = {0: 'Medium Spenders', 1: 'High Spenders', 2: 'High Spenders', 3: 'Low Spenders', 4: 'Low Spenders'}
    persona_map = {
        0: 'Mainstream Middle',
        1: 'Affluent VIPs',
        2: 'Young Enthusiasts',
        3: 'Careful Savers',
        4: 'Budget Conscious'
    }
    action_map = {
        0: 'Seasonal mass campaigns, loyalty point incentives, family retail packages.',
        1: 'Concierge access, luxury brand previews, personalized VIP gifting.',
        2: 'Fast-fashion launches, social media influencer activations, BNPL payment options.',
        3: 'Highlight product durability, premium warranty protection, high-value bundles.',
        4: 'Clearance alerts, discount vouchers, essentials and bulk-buy savings.'
    }
    
    return {
        'Predicted_Cluster': int(pred_cluster),
        'Spending_Group': tier_map[pred_cluster],
        'Persona': persona_map[pred_cluster],
        'Recommended_Action': action_map[pred_cluster]
    }

# Test with prospective customer profiles
test_cases = [
    {'Name': 'Lead Alice', 'Income': 92, 'Spend': 88},
    {'Name': 'Lead Bob', 'Income': 22, 'Spend': 18},
    {'Name': 'Lead Carol', 'Income': 58, 'Spend': 50},
    {'Name': 'Lead Dave', 'Income': 95, 'Spend': 15},
    {'Name': 'Lead Emma', 'Income': 28, 'Spend': 82}
]

inference_results = []
for tc in test_cases:
    res = predict_customer_segment(tc['Income'], tc['Spend'], kmeans)
    res['Customer'] = tc['Name']
    res['Income_k$'] = tc['Income']
    res['Spend_Score'] = tc['Spend']
    inference_results.append(res)

pd.DataFrame(inference_results)[['Customer', 'Income_k$', 'Spend_Score', 'Predicted_Cluster', 'Spending_Group', 'Persona', 'Recommended_Action']]


---
## Step 8: Strategic Business Recommendations (Capstone Presentation)

| Spending Group | Persona | Key Demographics | Behavioral Trait | Recommended Marketing & Retail Strategy |
| :--- | :--- | :--- | :--- | :--- |
| **High Spenders** | **Affluent VIPs** (Cluster 1) | Mean Age: 33 yrs<br>Income: $86.5k<br>Spend: 82.1 | Brand conscious, luxury buyers, highly responsive to prestige | Launch exclusive VIP loyalty lounges, invite-only designer runway shows, private shopping suites, and zero-friction concierge checkout. |
| **High Spenders** | **Young Enthusiasts** (Cluster 2) | Mean Age: 25 yrs<br>Income: $25.7k<br>Spend: 79.4 | Trend driven, impulsive, active on TikTok/Instagram, low income | Feature trendy fast-fashion and streetwear, student discount partnerships, pop-up events, and Buy-Now-Pay-Later (Klarna/Afterpay) integrations. |
| **Medium Spenders** | **Mainstream Middle** (Cluster 0) | Mean Age: 43 yrs<br>Income: $55.3k<br>Spend: 49.5 | Pragmatic, family-centric, values balanced utility (40.5% of mall traffic) | Host weekend family entertainment, dining vouchers, back-to-school expos, and tiered reward points redeemable across food & departmental stores. |
| **Low Spenders** | **Careful Savers** (Cluster 3) | Mean Age: 41 yrs<br>Income: $88.2k<br>Spend: 17.1 | High wealth, disciplined, low purchase frequency, value-seekers | Showcase investment-grade goods (electronics, timeless watches, home appliances), premium warranties, and emphasize quality over fleeting fashion. |
| **Low Spenders** | **Budget Conscious** (Cluster 4) | Mean Age: 45 yrs<br>Income: $26.3k<br>Spend: 20.9 | Price sensitive, strictly essential purchases, coupon driven | Position discount grocery outlets, seasonal end-of-quarter blowout sales, coupon booklets, and value bulk packs. |

---
### 🏁 Conclusion
By segmenting mall visitors into quantifiable, behavioral clusters, mall executives and retail partners can transition from indiscriminate generic broadcasts to hyper-targeted, high-conversion campaigns, driving both immediate revenue expansion and sustained customer loyalty.
